# TP1 — Analyse de Données Touristiques

Ce notebook s'appuie sur le package `tourisme/` (loader, analysis, visualizer).  
Il est également utilisable en ligne de commande :
```bash
python main.py report --file tourisme_brut.csv --output output/
```


## 1. Chargement et aperçu des données


In [ ]:
from tourisme import load_data, TourismeAnalyser, TourismeVisualizer

df = load_data("tourisme_brut.csv")
df.head()

In [ ]:
print(f"Shape : {df.shape}")
df.info()

In [ ]:
df.describe()

## 2. Rapport statistique textuel


In [ ]:
analyser = TourismeAnalyser(df)
analyser.print_report()

## 3. Visualisations statistiques

Les graphiques sont aussi générés en PNG via le CLI (`visualize`).  
Ici on les affiche directement dans le notebook.


In [ ]:
import matplotlib
matplotlib.use('inline')  # notebook backend
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

def fmt_k(x, _=None):
    if x >= 1_000_000: return f'{x/1_000_000:.1f}M'
    if x >= 1_000:     return f'{x/1_000:.0f}k'
    return str(int(x))


### 3.1 Visiteurs par région


In [ ]:
data = analyser.visitors_by_region()
fig, ax = plt.subplots(figsize=(10, max(4, len(data)*0.6)))
bars = ax.barh(data['region'], data['total_visiteurs'], color=sns.color_palette('muted'))
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.set_xlabel('Total visiteurs'); ax.set_title('Total des visiteurs par région', fontweight='bold')
ax.invert_yaxis()
for b in bars:
    ax.text(b.get_width()*1.01, b.get_y()+b.get_height()/2, fmt_k(b.get_width()), va='center', ha='left', fontsize=9)
plt.tight_layout(); plt.show()

### 3.2 Évolution annuelle


In [ ]:
data = analyser.visitors_by_year()
fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette('muted', len(data))
bars = ax.bar(data['annee'].astype(str), data['total_visiteurs'], color=colors)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.set_xlabel('Année'); ax.set_ylabel('Total visiteurs')
ax.set_title('Évolution annuelle des visiteurs', fontweight='bold')
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()*1.005, fmt_k(b.get_height()), ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

### 3.3 Saisonnalité mensuelle


In [ ]:
data = analyser.visitors_by_month()
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(data['mois_nom'], data['moy_visiteurs'], marker='o', linewidth=2.2, color='#2a7ae2')
ax.fill_between(data['mois_nom'], data['moy_visiteurs'], alpha=0.15, color='#2a7ae2')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_k))
ax.set_xlabel('Mois'); ax.set_ylabel('Visiteurs (moy.)')
ax.set_title('Saisonnalité mensuelle (moyenne toutes régions)', fontweight='bold')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

### 3.4 Dépense moyenne par région


In [ ]:
sp = analyser.spending_by_region()
if sp is not None:
    fig, ax = plt.subplots(figsize=(10, max(4, len(sp)*0.6)))
    bars = ax.barh(sp['region'], sp['depense_moy'], color=sns.color_palette('coolwarm_r', len(sp)))
    ax.set_xlabel('Dépense moyenne (€)'); ax.set_title('Dépense moyenne par région', fontweight='bold')
    ax.invert_yaxis()
    for b in bars:
        ax.text(b.get_width()*1.005, b.get_y()+b.get_height()/2, f'{b.get_width():.1f} €', va='center', ha='left', fontsize=9)
    plt.tight_layout(); plt.show()

### 3.5 Répartition des hébergements


In [ ]:
acc = analyser.accommodation_distribution()
if acc is not None:
    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        acc['total_visiteurs'], labels=acc['hebergement'],
        autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel')
    )
    [t.set_fontsize(10) for t in autotexts]
    ax.set_title('Répartition des visiteurs par type d\'hébergement', fontweight='bold')
    plt.tight_layout(); plt.show()

### 3.6 Heatmap région × mois


In [ ]:
import pandas as pd
pivot = df.groupby(['region','mois'])['visiteurs'].sum().unstack(fill_value=0)
pivot.columns = [f'Mois {m:02d}' for m in pivot.columns]
fig, ax = plt.subplots(figsize=(14, max(5, len(pivot)*0.7)))
sns.heatmap(pivot/1000, annot=True, fmt='.0f', linewidths=0.4,
            cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Visiteurs (milliers)'})
ax.set_title('Visiteurs par région et par mois (milliers)', fontweight='bold')
ax.set_ylabel('Région'); ax.set_xlabel('Mois')
plt.tight_layout(); plt.show()

## 4. Export PNG

Générer tous les graphiques en PNG dans le dossier `output/` :


In [ ]:
visualizer = TourismeVisualizer(analyser, output_dir='output')
saved = visualizer.generate_all()
print('\nFichiers créés :')
for p in saved:
    print(' ', p)